# Titanic Ride Ticket Classifier

Educational notebook version of the final project (teacher-friendly walkthrough).

**How to run:** Restart the kernel, then run cells top-to-bottom (Run All).

This notebook follows the same core rules as the Python submission:
- Ignore zero and missing fares when building class ranges
- Classify by best available class on overlap
- Estimate survival using `pclass + sex + age_group`
- Validate inputs (name rejects digits; fare validated against global dataset min/max)

In [1]:
import math
import random
from pathlib import Path

import pandas as pd

## Load dataset


In [2]:
# Load dataset (robust path handling)

NOTEBOOK_DIR = Path.cwd()

def resolve_csv_file() -> Path:
    candidates = [
        NOTEBOOK_DIR / "titanic.csv",
        NOTEBOOK_DIR / "homework" / "hw3" / "titanic.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Titanic dataset not found. Tried: " + ", ".join(str(p) for p in candidates)
    )

CSV_FILE = resolve_csv_file()
df = pd.read_csv(CSV_FILE)

print("Loaded:", CSV_FILE.resolve())
print("Shape:", df.shape)
df.head()

Loaded: C:\dev\amdocs-ai-course\homework\hw3\titanic.csv
Shape: (1309, 14)


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Prepare fare ranges and existing tickets

Ignore zero and missing fares for class-range analysis.


In [3]:
REQUIRED_COLUMNS = {"pclass", "survived", "sex", "age", "ticket", "fare"}

def validate_titanic_schema(df: pd.DataFrame) -> None:
    missing = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing:
        raise ValueError("Titanic dataset is missing required columns: " + ", ".join(missing))


def get_global_fare_limits(df: pd.DataFrame) -> tuple[float, float]:
    fare = pd.to_numeric(df["fare"], errors="coerce")
    fare = fare[fare.notna() & (fare > 0)]
    if fare.empty:
        raise ValueError("Dataset has no valid positive fares to validate against.")
    return float(fare.min()), float(fare.max())


def get_fare_ranges(df: pd.DataFrame) -> dict[int, tuple[float, float]]:
    fare_df = df[["pclass", "fare"]].copy()
    fare_df["fare"] = pd.to_numeric(fare_df["fare"], errors="coerce")
    fare_df["pclass"] = pd.to_numeric(fare_df["pclass"], errors="coerce")
    fare_df = fare_df[fare_df["fare"].notna() & (fare_df["fare"] > 0)].copy()
    fare_df = fare_df[fare_df["pclass"].notna()].copy()

    fare_ranges: dict[int, tuple[float, float]] = {}
    for pclass in sorted(fare_df["pclass"].unique()):
        class_fares = fare_df.loc[fare_df["pclass"] == pclass, "fare"].dropna()
        if class_fares.empty:
            continue
        fare_ranges[int(pclass)] = (float(class_fares.min()), float(class_fares.max()))
    if not fare_ranges:
        raise ValueError("Could not compute class fare ranges (no valid fares found).")
    return fare_ranges


def get_existing_tickets(df: pd.DataFrame) -> set[str]:
    return set(df["ticket"].dropna().astype(str).str.strip())


validate_titanic_schema(df)
global_min, global_max = get_global_fare_limits(df)
fare_ranges = get_fare_ranges(df)
existing_tickets = get_existing_tickets(df)

print("Global fare range:", (global_min, global_max))
fare_ranges

Global fare range: (3.1708, 512.3292)


{1: (5.0, 512.3292), 2: (9.6875, 73.5), 3: (3.1708, 69.55)}

In [4]:
NAME_ALLOWED_SEPARATORS = set(" -'")

def is_valid_name(name: str) -> bool:
    name = name.strip()
    if not name:
        return False
    if any(ch.isdigit() for ch in name):
        return False
    if not any(ch.isalpha() for ch in name):
        return False
    if not all(ch.isalpha() or ch in NAME_ALLOWED_SEPARATORS for ch in name):
        return False
    return True


def get_valid_name() -> str:
    while True:
        name = input("Enter passenger name: ").strip()
        if is_valid_name(name):
            return name
        print("Name is invalid. Use letters only (spaces/-/'). No digits.")


def get_valid_age() -> float:
    while True:
        raw_age = input("Enter passenger age: ").strip()
        try:
            age = float(raw_age)
            if not math.isfinite(age):
                raise ValueError
            if 0 <= age <= 130:
                return age
            print("Age is wrong. Please enter age between 0 and 130.")
        except ValueError:
            print("Age must be a number. Please try again.")


def get_valid_sex() -> str:
    while True:
        sex = input("Enter passenger sex (male/female): ").strip().lower()
        if sex in {"male", "female"}:
            return sex
        print("Sex value is illegal. Please enter male or female.")


def get_valid_fare(global_min: float, global_max: float) -> float:
    while True:
        raw_fare = input(f"Enter passenger fare (${global_min:g} - ${global_max:g}): ").strip()
        try:
            fare = float(raw_fare)
        except ValueError:
            print("Fare must be a float number. Please try again.")
            continue
        if not math.isfinite(fare):
            print("Fare must be a float number. Please try again.")
            continue
        if global_min <= fare <= global_max:
            return fare
        print("Your fare payment is illegal. Please pay again.")

## Class assignment

If a fare matches more than one range, choose the best class.


In [5]:
def classify_fare_to_class(fare: float, fare_ranges: dict) -> int:
    matching_classes = [
        pclass
        for pclass, (min_fare, max_fare) in fare_ranges.items()
        if min_fare <= fare <= max_fare
    ]

    if matching_classes:
        return min(matching_classes)

    closest_class = min(
        fare_ranges,
        key=lambda pclass: min(
            abs(fare - fare_ranges[pclass][0]),
            abs(fare - fare_ranges[pclass][1])
        )
    )
    return closest_class


## Ticket generation and file writing


In [6]:
def generate_unique_ticket(existing_tickets: set[str]) -> str:
    while True:
        ticket_no = str(random.randint(100000, 999999))
        if ticket_no not in existing_tickets:
            existing_tickets.add(ticket_no)
            return ticket_no


def save_ticket(
    ticket_no: str,
    fare: float,
    age: float,
    pclass: int,
    sex: str,
    name: str,
    output_path: Path,
) -> None:
    line = "-" * 50

    def row(left: str, right: str) -> str:
        return f"| {left:<24} | {right:<19} |"

    ticket_text = "\n".join(
        [
            line,
            row(f"ticket: {ticket_no}", f" fare: {fare:g}"),
            line,
            row(f"age: {age:g}", f" class: {pclass}"),
            line,
            row(f"sex: {sex}", f" name: {name}"),
            line,
        ]
    )

    output_path.write_text(ticket_text, encoding="utf-8")

## Survival calculation

Use `pclass + sex + age_group`, then fallback if an exact subgroup is missing.


In [7]:
def calculate_survival(df: pd.DataFrame, pclass: int, sex: str, age: float) -> tuple[float, float]:
    temp_df = df.copy()
    temp_df["age"] = pd.to_numeric(temp_df["age"], errors="coerce")
    temp_df["survived"] = pd.to_numeric(temp_df["survived"], errors="coerce")
    temp_df["age_group"] = temp_df["age"].apply(
        lambda x: "under18" if pd.notna(x) and x < 18 else "over18"
    )

    passenger_age_group = "under18" if age < 18 else "over18"

    level_1 = temp_df[
        (temp_df["pclass"] == pclass) &
        (temp_df["sex"] == sex) &
        (temp_df["age_group"] == passenger_age_group)
    ]
    if not level_1.empty:
        survival_pct = round(level_1["survived"].mean() * 100, 1)
        return survival_pct, round(100 - survival_pct, 1)

    level_2 = temp_df[
        (temp_df["pclass"] == pclass) &
        (temp_df["sex"] == sex)
    ]
    if not level_2.empty:
        survival_pct = round(level_2["survived"].mean() * 100, 1)
        return survival_pct, round(100 - survival_pct, 1)

    level_3 = temp_df[temp_df["pclass"] == pclass]
    if not level_3.empty:
        survival_pct = round(level_3["survived"].mean() * 100, 1)
        return survival_pct, round(100 - survival_pct, 1)

    survival_pct = round(temp_df["survived"].mean() * 100, 1)
    return survival_pct, round(100 - survival_pct, 1)


## Main program

Run this cell to start the interactive system.


In [8]:
def main() -> None:
    name = get_valid_name()
    age = get_valid_age()
    sex = get_valid_sex()
    fare = get_valid_fare(global_min, global_max)

    pclass = classify_fare_to_class(fare, fare_ranges)
    ticket_no = generate_unique_ticket(existing_tickets)

    output_file = NOTEBOOK_DIR / "passenger_ticket.txt"
    save_ticket(ticket_no, fare, age, pclass, sex, name, output_file)

    _survival_pct, death_pct = calculate_survival(df, pclass, sex, age)
    first_name = name.split()[0] if name.split() else name

    print("\n" + "=" * 55)
    print("Passenger registered successfully.")
    print(f"Assigned class: {pclass}")
    print(f"New ticket number: {ticket_no}")
    print(f"Ticket saved to '{output_file.name}'")
    print(f"\nDear {first_name}, your chances to die on our trip are {death_pct}%.")
    print("Enjoy your trip ☺")
    print("=" * 55)

In [1]:
main()


NameError: name 'main' is not defined